# 05: SHAP Explainability
This notebook loads the trained lung classification model and uses SHAP (SHapley Additive exPlanations) to explain the feature importance both globally and for individual patient predictions.

In [ ]:
# Uncomment to install shap if not installed
# !pip install -q shap

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import tensorflow as tf
from sklearn.model_selection import train_test_split

# Paths
data_dir = r'C:\Users\Kruthi K Shetty\data-science\data'
models_dir = r'C:\Users\Kruthi K Shetty\data-science\models'
outputs_dir = r'C:\Users\Kruthi K Shetty\data-science\outputs\shap'

os.makedirs(outputs_dir, exist_ok=True)

In [ ]:
# 1. Load Data
print("Loading model and data...")
model_path = os.path.join(models_dir, 'lung_model.keras')
model = tf.keras.models.load_model(model_path)

df_path = os.path.join(data_dir, 'features_df.csv')
df = pd.read_csv(df_path)

test_set_path = os.path.join(data_dir, 'test_set.pkl')
with open(test_set_path, 'rb') as f:
    test_data = pickle.load(f)
scaler = test_data['scaler']

le_path = os.path.join(models_dir, 'label_encoder.pkl')
with open(le_path, 'rb') as f:
    le = pickle.load(f)

# Re-run train_test_split on dataframe to keep patient IDs
meta_cols = ['npy_index', 'patient_id', 'filename', 'cycle_start', 'cycle_end', 'label']
feature_cols = [c for c in df.columns if c not in meta_cols]

X = df[feature_cols].values
y_text = df['label'].values
y_encoded = le.transform(y_text)

# We use the exact same random_state=42 as Notebook 4 to get the exact same split
indices = np.arange(len(df))
X_train_raw, X_test_raw, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y_encoded, indices, test_size=0.2, stratify=y_encoded, random_state=42
)

df_test = df.iloc[idx_test].copy()
df_test['true_label'] = y_text[idx_test]

# Scale features (using the loaded scaler)
X_train_scaled = scaler.transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

print(f"Test set shape: {X_test_scaled.shape}")

In [ ]:
# Generate predictions to find correct ones for Plot C
y_pred_probs = model.predict(X_test_scaled)
y_pred = np.argmax(y_pred_probs, axis=1)
df_test['pred_label'] = le.inverse_transform(y_pred)
df_test['is_correct'] = df_test['true_label'] == df_test['pred_label']

In [ ]:
# 2. Initialize SHAP Explainer
print("Initializing SHAP Explainer...")

try:
    # Try DeepExplainer first (faster for NNs)
    print("Trying DeepExplainer...")
    background_sample = shap.sample(X_train_scaled, 100)
    explainer = shap.DeepExplainer(model, background_sample)
    shap_values = explainer.shap_values(X_test_scaled)
except Exception as e:
    print(f"DeepExplainer failed: {e}")
    print("Falling back to KernelExplainer...")
    background = shap.kmeans(X_train_scaled, 100)
    # KernelExplainer wrapper to predict probabilities
    explainer = shap.KernelExplainer(model.predict, background)
    # KernelExplainer is slow, so compute on a smaller sample
    sample_idx = np.random.choice(X_test_scaled.shape[0], min(200, X_test_scaled.shape[0]), replace=False)
    X_test_scaled = X_test_scaled[sample_idx]
    df_test = df_test.iloc[sample_idx]
    shap_values = explainer.shap_values(X_test_scaled)

# Ensure shap_values is a list of arrays (one for each class) for plotting consistency
if not isinstance(shap_values, list):
    if len(shap_values.shape) == 3:
        shap_values = [shap_values[:, :, i] for i in range(shap_values.shape[2])]

In [ ]:
# 3. Plot A - Global Feature Importance
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_scaled, feature_names=feature_cols, class_names=le.classes_, plot_type="bar", show=False)
plt.title("Global feature importance across all classes")
plt.tight_layout()
plt.savefig(os.path.join(outputs_dir, 'global_importance.png'), dpi=300)
plt.show()

# Beeswarm plot for 'Crackle' (Index 1 usually)
crackle_idx = list(le.classes_).index('Crackle')
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values[crackle_idx], X_test_scaled, feature_names=feature_cols, plot_type="dot", show=False)
plt.title("Beeswarm Feature Importance for class 'Crackle'")
plt.tight_layout()
plt.savefig(os.path.join(outputs_dir, 'crackle_beeswarm.png'), dpi=300)
plt.show()

In [ ]:
# Plot B - Per-class bar chart
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

for i, class_name in enumerate(le.classes_):
    plt.sca(axes[i])
    mean_abs_shap = np.abs(shap_values[i]).mean(axis=0)
    
    top_indices = np.argsort(mean_abs_shap)[-10:]
    top_features = [feature_cols[j] for j in top_indices]
    top_shap = mean_abs_shap[top_indices]
    
    axes[i].barh(top_features, top_shap, color='skyblue')
    axes[i].set_title(f"Class: {class_name}")
    axes[i].set_xlabel("Mean Absolute SHAP Value")

plt.tight_layout()
plt.savefig(os.path.join(outputs_dir, 'per_class_importance.png'), dpi=300)
plt.show()

In [ ]:
# Plot C - Individual patient waterfall plots
targets = ['Normal', 'Wheeze', 'Crackle']
selected_indices = []

for t in targets:
    candidates = df_test[(df_test['true_label'] == t) & (df_test['is_correct'] == True)]
    if not candidates.empty:
        idx_in_df_test = candidates.index[0]
        pos_idx = np.where(df_test.index == idx_in_df_test)[0][0]
        selected_indices.append((t, pos_idx, candidates.iloc[0]['patient_id']))

# Extract expected value (base value) safely
if isinstance(explainer.expected_value, np.ndarray):
    expected_values = explainer.expected_value
elif isinstance(explainer.expected_value, list):
    expected_values = explainer.expected_value
else:
    expected_values = [explainer.expected_value] * len(le.classes_)

for class_name, pos_idx, patient_id in selected_indices:
    class_idx = list(le.classes_).index(class_name)
    
    base_val = expected_values[class_idx]
    if isinstance(base_val, np.ndarray) and base_val.size >= 1:
        base_val = base_val[class_idx] if base_val.size > 1 else base_val[0]
        
    shap_vals = shap_values[class_idx][pos_idx]
    data_vals = X_test_scaled[pos_idx]
    
    exp = shap.Explanation(
        values=shap_vals,
        base_values=float(base_val),
        data=data_vals,
        feature_names=feature_cols
    )
    
    plt.figure(figsize=(10, 6))
    shap.waterfall_plot(exp, show=False)
    plt.title(f"Patient {patient_id} — Predicted: {class_name}, Actual: {class_name}")
    plt.tight_layout()
    plt.savefig(os.path.join(outputs_dir, f'waterfall_patient_{patient_id}_{class_name}.png'), bbox_inches='tight', dpi=300)
    plt.show()

In [ ]:
# 4. Top 5 Features Overall
print("Top 5 Overall Features and Acoustic Meaning:")

mean_abs_shap_overall = np.zeros(len(feature_cols))
for i in range(len(le.classes_)):
    mean_abs_shap_overall += np.abs(shap_values[i]).mean(axis=0)

top_5_idx = np.argsort(mean_abs_shap_overall)[-5:][::-1]
top_5_features = [feature_cols[i] for i in top_5_idx]

feature_meanings = {
    'mfcc': 'Mel-Frequency Cepstral Coefficients: represents the overall shape of the spectral envelope (timbre).',
    'chroma': 'Chroma Features: relates to the pitch classes present in the audio.',
    'mel': 'Mel Spectrogram: energy in different frequency bands scaled to human hearing.',
    'contrast': 'Spectral Contrast: difference in amplitude between peaks and valleys in the spectrum.',
    'tonnetz': 'Tonal Centroid Features: estimates the harmonic content.',
    'zcr': 'Zero Crossing Rate: rate at which the signal changes sign (good for identifying noisy/crackling sounds).',
    'spectral_centroid': 'Spectral Centroid: the "center of mass" of the spectrum (brightness of sound).',
    'spectral_rolloff': 'Spectral Rolloff: frequency below which a specified percentage of total spectral energy lies.',
    'rmse': 'Root Mean Square Energy: the overall volume or energy of the audio.'
}

for rank, feat_name in enumerate(top_5_features, 1):
    meaning = "Specific acoustic feature"
    for key, desc in feature_meanings.items():
        if key in feat_name.lower():
            meaning = desc
            break
    print(f"{rank}. {feat_name}: {meaning}")